<a href="https://colab.research.google.com/github/ABHAY-SINGH-CODER/ABHAY-SINGH-CODER/blob/main/LastGroundDinoTry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ========================
# Install CPU-only PyTorch + Dependencies
# ========================

# 1. CPU-only PyTorch
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# 2. Basic dependencies
!pip install opencv-python matplotlib numpy pillow tqdm

# 3. GroundingDINO
!pip install git+https://github.com/IDEA-Research/GroundingDINO.git

# 4. Segment Anything Model (SAM)
!pip install git+https://github.com/facebookresearch/segment-anything.git

# 5. CLIP
!pip install git+https://github.com/openai/CLIP.git

# 6. Transformers (if needed for text processing in CLIP/DINO)
!pip install transformers



Looking in indexes: https://download.pytorch.org/whl/cpu
  Cloning https://github.com/IDEA-Research/GroundingDINO.git to /tmp/pip-req-build-q57wflxt
  Running command git clone --filter=blob:none --quiet https://github.com/IDEA-Research/GroundingDINO.git /tmp/pip-req-build-q57wflxt
  Resolved https://github.com/IDEA-Research/GroundingDINO.git to commit 856dde20aee659246248e20734ef9ba5214f5e44
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-vo9v4h0a
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-vo9v4h0a
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-rk3bv0d5
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git

In [2]:
# ======================
# Minimal CPU-Only GroundingDINO Inference (returns image with boxes)
# ======================

import torch
import cv2
from groundingdino.util.inference import load_model, load_image, predict, annotate

# -------- SETTINGS --------
image_path = "/content/my_image.png"
prompt_text = "vendor selling vegetables"
model_config_path = "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
model_weights_path = "/content/GroundingDINO/weights/groundingdino_swint_ogc.pth"

# -------- LOAD MODEL --------
device = "cpu"
model = load_model(model_config_path, model_weights_path)
model = model.to(device)

# -------- READ AND PROCESS IMAGE --------
image_source, image_tensor = load_image(image_path)

# -------- RUN PREDICTION --------
boxes, logits, phrases = predict(
    model=model,
    image=image_tensor,
    caption=prompt_text,
    box_threshold=0.35,
    text_threshold=0.25,
    device=device
)

# -------- ANNOTATE (pass logits too) --------
annotated_frame = annotate(image_source, boxes, logits, phrases)

# -------- RETURN / SAVE IMAGE --------
output_path = "/content/annotated_image.jpg"
cv2.imwrite(output_path, annotated_frame)
print(f"Saved annotated image at {output_path}")


final text_encoder_type: bert-base-uncased


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


Saved annotated image at /content/annotated_image.jpg


In [6]:



!cp /content/sam_vit_h_4b8939.pth


cp: missing destination file operand after '/content/sam_vit_h_4b8939.pth'
Try 'cp --help' for more information.


In [3]:
# ======================
# GroundingDINO -> SAM (CPU) pipeline
# ======================

import os
import torch
import cv2
import numpy as np
from groundingdino.util.inference import load_model, load_image, predict, annotate

# Try to import SAM
try:
    from segment_anything import sam_model_registry, SamPredictor
except Exception as e:
    raise ImportError("segment_anything not found. Install it with: pip install git+https://github.com/facebookresearch/segment-anything.git") from e

# -------- SETTINGS (change as needed) --------
image_path = "/content/my_image.png"
prompt_text = "vendor selling vegetables"
gd_config = "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
gd_weights = "/content/GroundingDINO/weights/groundingdino_swint_ogc.pth"
sam_checkpoint = "/content/sam_vit_h_4b8939.pth"   # change if you have a different SAM checkpoint
device = "cpu"  # CPU-only

# -------- Ensure SAM checkpoint exists (download if not) --------
if not os.path.exists(sam_checkpoint):
    print(f"[INFO] SAM checkpoint not found at {sam_checkpoint}. Downloading (this may take time)...")
    # ViT-H checkpoint (official): ~345MB
    !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O {sam_checkpoint}
    print("[INFO] Download complete.")

# -------- Load GroundingDINO model (CPU) --------
print("[INFO] Loading GroundingDINO model...")
model = load_model(gd_config, gd_weights, device=device)
model = model.to(device)
print("[INFO] GroundingDINO loaded.")

# -------- Load image via helper --------
print("[INFO] Reading image...")
image_source, image_tensor = load_image(image_path)  # image_source: numpy RGB, image_tensor: processed input for DINO

# -------- Run GroundingDINO prediction --------
print("[INFO] Running GroundingDINO...")
boxes, logits, phrases = predict(
    model=model,
    image=image_tensor,
    caption=prompt_text,
    box_threshold=0.35,
    text_threshold=0.25,
    device=device
)
print(f"[INFO] GroundingDINO returned {len(boxes)} boxes, phrases: {phrases}")

# Fallback if no boxes found
if boxes is None or len(boxes) == 0:
    h, w = image_source.shape[:2]
    boxes = np.array([[0, 0, w-1, h-1]])
    print("[WARN] No boxes found — using full-image box as fallback.")

# -------- Annotate DINO boxes (RGB numpy) --------
annotated_dino = annotate(image_source.copy(), boxes, logits, phrases)

# -------- Load SAM and create predictor --------
print("[INFO] Loading SAM model...")
sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)  # change "vit_h" to "vit_b" if using a smaller checkpoint
sam.to(device)
predictor = SamPredictor(sam)
print("[INFO] SAM loaded. Setting image for predictor (this may compute image embeddings)...")
predictor.set_image(image_source)  # expects RGB numpy image

# -------- Prepare boxes for SAM --------
# SAM expects boxes as torch tensor in (x0,y0,x1,y1) in image coordinate space.
# Use predictor.transform to convert.
boxes_torch = predictor.transform.apply_boxes_torch(
    torch.tensor(boxes, dtype=torch.float32), image_source.shape[:2]
)  # shape: (N,4)

# Move boxes to device (CPU)
boxes_torch = boxes_torch.to(device)

# -------- Run SAM prediction using boxes as prompts --------
print("[INFO] Running SAM predictor on boxes (this may be slow on CPU)...")
with torch.no_grad():
    masks_torch, scores, logits_mask = predictor.predict_torch(
        point_coords=None,
        point_labels=None,
        boxes=boxes_torch,
        multimask_output=False,
    )
# masks_torch shape: (N, 1, H, W) (for multimask_output=False)

# Convert masks to numpy boolean masks (N, H, W)
masks = [m[0].cpu().numpy().astype(bool) for m in masks_torch]

print(f"[INFO] SAM returned {len(masks)} mask(s).")

# -------- Overlay masks on annotated image --------
overlay = annotated_dino.copy()  # RGB
h, w = overlay.shape[:2]
mask_overlay = overlay.copy()

# Choose colors for masks (here a single green-ish color)
for i, m in enumerate(masks):
    color = np.array([0, 200, 0], dtype=np.uint8)  # RGB
    # Create colored mask image
    colored_mask = np.zeros_like(mask_overlay, dtype=np.uint8)
    colored_mask[m] = color
    # Blend
    mask_overlay = cv2.addWeighted(mask_overlay, 1.0, colored_mask, 0.35, 0)

# Optionally draw boundaries of masks
for m in masks:
    # find contours on uint8 mask
    mask_uint8 = (m.astype(np.uint8) * 255)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(mask_overlay, contours, -1, (0,255,0), 2)  # RGB green contour

# Convert to BGR for OpenCV saving (opencv uses BGR)
final_bgr = cv2.cvtColor(mask_overlay, cv2.COLOR_RGB2BGR)

# -------- Save final image --------
output_path = "/content/annotated_with_sam.jpg"
cv2.imwrite(output_path, final_bgr)
print(f"[DONE] Saved final image with DINO boxes + SAM masks at: {output_path}")

# Also expose variables for further use in the notebook
result = {
    "dino_boxes": boxes,
    "dino_phrases": phrases,
    "sam_masks": masks,
    "annotated_image_rgb": mask_overlay,
    "annotated_image_bgr_path": output_path
}


[INFO] SAM checkpoint not found at /content/sam_vit_h_4b8939.pth. Downloading (this may take time)...
/content/sam_vit_h_ 100%[===================>]   2.39G  70.6MB/s    in 26s     
[INFO] Download complete.
[INFO] Loading GroundingDINO model...
final text_encoder_type: bert-base-uncased
[INFO] GroundingDINO loaded.
[INFO] Reading image...
[INFO] Running GroundingDINO...
[INFO] GroundingDINO returned 2 boxes, phrases: ['vegetables', 'vendor']
[INFO] Loading SAM model...
[INFO] SAM loaded. Setting image for predictor (this may compute image embeddings)...


[INFO] Running SAM predictor on boxes (this may be slow on CPU)...
[INFO] SAM returned 2 mask(s).
[DONE] Saved final image with DINO boxes + SAM masks at: /content/annotated_with_sam.jpg


In [1]:
# ============================
# GroundingDINO + SAM + CLIP pipeline (Colab-ready)
# ============================
# Paste entire cell into Google Colab and run.
# It will:
#  - install minimal dependencies (if missing)
#  - clone GroundingDINO repo
#  - download checkpoints if not present
#  - let you upload an image, run DINO -> CLIP filtering -> merge -> SAM
#  - save result to /content/combined_result.jpg

# ------------- INSTALL (only if needed) -------------
# Uncomment the installs below if you don't already have packages installed.
# They can take a few minutes to install.
# !pip -q install --upgrade pip
# !pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !pip -q install opencv-python matplotlib numpy pillow tqdm
# !pip -q install git+https://github.com/IDEA-Research/GroundingDINO.git
# !pip -q install git+https://github.com/facebookresearch/segment-anything.git
# !pip -q install git+https://github.com/openai/CLIP.git

import os
import sys
import torch
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
from google.colab import files

# ------------- Clone GroundingDINO repo (if not present) -------------
if not os.path.exists("GroundingDINO"):
    print("[SETUP] Cloning GroundingDINO repo...")
    !git clone --depth 1 https://github.com/IDEA-Research/GroundingDINO.git

# Add repo to path so we can import its helpers
sys.path.append("GroundingDINO")

# ------------- Imports (after repo present) -------------
try:
    from groundingdino.util.inference import load_model, load_image, predict, annotate
except Exception as e:
    raise ImportError("Could not import groundingdino utilities. Ensure the repo was cloned correctly.") from e

try:
    from segment_anything import sam_model_registry, SamPredictor
except Exception as e:
    raise ImportError("segment_anything not available. Install it: pip install git+https://github.com/facebookresearch/segment-anything.git") from e

try:
    import clip
except Exception as e:
    raise ImportError("CLIP not available. Install it: pip install git+https://github.com/openai/CLIP.git") from e

# ------------- Settings (edit these) -------------
# Upload your local image when prompted (or set image_path to an existing file in /content)
print("Upload your image (choose file from local).")
uploaded = files.upload()  # choose file(s)
if len(uploaded) == 0:
    raise RuntimeError("No file uploaded.")
image_path = list(uploaded.keys())[0]

# Example prompts (edit as needed)
dino_caption = "vendor, vegetables"            # prompt passed to GroundingDINO (gets candidate boxes)
merge_text = "vendor with vegetables"          # combined concept for CLIP scoring
clip_threshold = 0.25                          # similarity threshold for selecting boxes (tune as needed)

# Model checkpoint paths (will be downloaded if missing)
gd_config = "/content/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
gd_checkpoint = "/content/groundingdino_swint_ogc.pth"
sam_checkpoint_h = "/content/sam_vit_h_4b8939.pth"   # large; use vit_b if you prefer smaller
sam_checkpoint_b = "/content/sam_vit_b_01ec64.pth"   # smaller checkpoint alternative

# Choose SAM checkpoint: prefer vit_b for CPU speed if vit_h not present
sam_checkpoint = sam_checkpoint_h if os.path.exists(sam_checkpoint_h) else sam_checkpoint_b

# ------------- Download checkpoints if missing -------------
# GroundingDINO checkpoint
if not os.path.exists(gd_checkpoint):
    print("[SETUP] Downloading GroundingDINO checkpoint (will take a while)...")
    !wget -q --show-progress https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth -O {gd_checkpoint}
    print("[SETUP] GroundingDINO checkpoint downloaded.")

# SAM checkpoint: download smaller ViT-B if ViT-H not desired or missing
if not os.path.exists(sam_checkpoint):
    # Try to fetch ViT-B by default (smaller)
    print("[SETUP] SAM checkpoint not found. Downloading SAM ViT-B (faster on CPU)...")
    !wget -q --show-progress https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O {sam_checkpoint}
    print("[SETUP] SAM checkpoint downloaded.")

# ------------- Device -------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

# ------------- Load GroundingDINO -------------
print("[INFO] Loading GroundingDINO model (this may take a while on CPU)...")
gd_model = load_model(gd_config, gd_checkpoint, device=device)
gd_model = gd_model.to(device)
print("[INFO] GroundingDINO loaded.")

# ------------- Load CLIP -------------
print("[INFO] Loading CLIP model...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("[INFO] CLIP loaded.")

# ------------- Load SAM -------------
# Choose registry key based on checkpoint file name
if "vit_h" in sam_checkpoint or "vit_h" in Path(sam_checkpoint).stem:
    sam_type = "vit_h"
elif "vit_b" in sam_checkpoint or "vit_b" in Path(sam_checkpoint).stem:
    sam_type = "vit_b"
else:
    # default to vit_b
    sam_type = "vit_b"

print(f"[INFO] Building SAM model ({sam_type})...")
sam = sam_model_registry[sam_type](checkpoint=sam_checkpoint)
sam.to(device)
predictor = SamPredictor(sam)
print("[INFO] SAM loaded.")

# ------------- Read image via GroundingDINO helper -------------
print("[INFO] Reading image:", image_path)
image_source, image_tensor = load_image(image_path)  # image_source: RGB numpy, image_tensor: preproc tensor for DINO
H, W = image_source.shape[:2]

# ------------- Run GroundingDINO to get candidate boxes -------------
print("[INFO] Running GroundingDINO inference...")
boxes, logits, phrases = predict(
    model=gd_model,
    image=image_tensor,
    caption=dino_caption,
    box_threshold=0.35,
    text_threshold=0.25,
    device=device
)
# boxes: numpy array (N,4) but sometimes normalized to [0,1] — handle both cases below

if boxes is None or len(boxes) == 0:
    print("[WARN] No boxes returned by GroundingDINO. Exiting.")
    result_path = "/content/combined_result.jpg"
    cv2.imwrite(result_path, cv2.cvtColor(image_source, cv2.COLOR_RGB2BGR))
    print(f"[DONE] Saved original image at {result_path}")
    raise SystemExit()

boxes_np = np.array(boxes)  # assume boxes as numpy already
# If boxes look normalized (values <=1), scale to pixel coords.
if boxes_np.max() <= 1.01:
    boxes_pixels = (boxes_np * np.array([W, H, W, H])).astype(int)
else:
    boxes_pixels = boxes_np.astype(int)

print(f"[INFO] GroundingDINO returned {len(boxes_pixels)} boxes.")

# ------------- Use CLIP to score each DINO crop against the merged text -------------
print("[INFO] Scoring DINO crops with CLIP against:", merge_text)
merge_text_tokens = clip.tokenize([merge_text]).to(device)
with torch.no_grad():
    merge_text_feat = clip_model.encode_text(merge_text_tokens)
    merge_text_feat = merge_text_feat / merge_text_feat.norm(dim=-1, keepdim=True)

selected_indices = []
selected_similarities = []
for i, (x0, y0, x1, y1) in enumerate(boxes_pixels):
    # clamp coords
    x0c, y0c = max(0, x0), max(0, y0)
    x1c, y1c = min(W - 1, x1), min(H - 1, y1)
    if x1c <= x0c or y1c <= y0c:
        continue

    crop_pil = Image.fromarray(image_source[y0c:y1c, x0c:x1c])
    try:
        crop_t = clip_preprocess(crop_pil).unsqueeze(0).to(device)
    except Exception as e:
        # in case crop is too small or invalid
        continue

    with torch.no_grad():
        img_feat = clip_model.encode_image(crop_t)
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
        sim = (img_feat @ merge_text_feat.T).item()
    # collect if above threshold
    if sim >= clip_threshold:
        selected_indices.append(i)
        selected_similarities.append(sim)

print(f"[INFO] Selected {len(selected_indices)} boxes above CLIP threshold {clip_threshold}.")

# ------------- If none selected, optionally relax threshold or select the top-k -------------
if len(selected_indices) == 0:
    # fallback: pick top-2 by similarity among all boxes (compute similarities)
    print("[WARN] No boxes passed threshold — selecting top 2 by similarity as fallback.")
    sims = []
    for i, (x0, y0, x1, y1) in enumerate(boxes_pixels):
        x0c, y0c = max(0, x0), max(0, y0)
        x1c, y1c = min(W - 1, x1), min(H - 1, y1)
        if x1c <= x0c or y1c <= y0c:
            sims.append(-1.0)
            continue
        crop_pil = Image.fromarray(image_source[y0c:y1c, x0c:x1c])
        try:
            crop_t = clip_preprocess(crop_pil).unsqueeze(0).to(device)
        except:
            sims.append(-1.0)
            continue
        with torch.no_grad():
            img_feat = clip_model.encode_image(crop_t)
            img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
            sim = (img_feat @ merge_text_feat.T).item()
        sims.append(sim)
    sims = np.array(sims)
    topk = min(2, len(sims))
    if topk > 0:
        top_idxs = sims.argsort()[-topk:][::-1].tolist()
        selected_indices = [int(i) for i in top_idxs if sims[i] > -0.5]
        selected_similarities = [float(sims[i]) for i in selected_indices]
    print(f"[INFO] Fallback selected indices: {selected_indices}")

# ------------- Merge selected boxes into a single union box -------------
if len(selected_indices) == 0:
    print("[WARN] Still no boxes to merge. Saving original image and exiting.")
    result_path = "/content/combined_result.jpg"
    cv2.imwrite(result_path, cv2.cvtColor(image_source, cv2.COLOR_RGB2BGR))
    print(f"[DONE] Saved original image at {result_path}")
    raise SystemExit()

selected_boxes = boxes_pixels[selected_indices]
x_min = int(selected_boxes[:, 0].min())
y_min = int(selected_boxes[:, 1].min())
x_max = int(selected_boxes[:, 2].max())
y_max = int(selected_boxes[:, 3].max())
merged_box = np.array([x_min, y_min, x_max, y_max], dtype=int)
print(f"[INFO] Merged box (pixels): {merged_box}")

# ------------- Use SAM to segment the merged box region -------------
print("[INFO] Running SAM on merged box to get mask (this computes image embeddings; may be slow on CPU).")
predictor.set_image(image_source)  # compute embeddings (cached in predictor)
# SAM expects boxes as (x0,y0,x1,y1) numpy in image coords, but predictor.predict_torch wants transformed boxes
boxes_for_sam = torch.tensor([merged_box], dtype=torch.float32)  # shape (1,4) in pixel coords
# Transform boxes to predictor coordinate space
boxes_transformed = predictor.transform.apply_boxes_torch(boxes_for_sam, image_source.shape[:2]).to(device)

with torch.no_grad():
    masks_t, scores_t, logits_t = predictor.predict_torch(
        point_coords=None,
        point_labels=None,
        boxes=boxes_transformed,
        multimask_output=False,
    )
# masks_t shape: (N,1,H,W) -> convert to boolean mask
masks_list = [m[0].cpu().numpy().astype(bool) for m in masks_t]
mask = masks_list[0]  # boolean HxW mask

# ------------- Prepare visualization overlay -------------
overlay = image_source.copy()  # RGB
# Create colored mask and composite
color = np.array([0, 200, 0], dtype=np.uint8)  # green
colored_mask = np.zeros_like(overlay, dtype=np.uint8)
colored_mask[mask] = color
composited = cv2.addWeighted(overlay, 1.0, colored_mask, 0.35, 0)

# Draw contours around mask
mask_uint8 = (mask.astype(np.uint8) * 255)
contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(composited, contours, -1, (0,255,0), 2)

# Draw merged bounding box and a label with CLIP score summary
cv2.rectangle(composited, (x_min, y_min), (x_max, y_max), (255,0,0), 3)  # blue box
label = f"Combined ({len(selected_indices)} regions) "
if len(selected_similarities) > 0:
    label += f"score={np.mean(selected_similarities):.2f}"
cv2.putText(composited, label, (x_min, max(15, y_min-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,0,0), 2, cv2.LINE_AA)

# ------------- Save result -------------
result_path = "/content/combined_result.jpg"
# convert RGB -> BGR for saving with cv2
cv2.imwrite(result_path, cv2.cvtColor(composited, cv2.COLOR_RGB2BGR))
print(f"[DONE] Saved combined detection + SAM mask to: {result_path}")

# Variables exposed for further analysis
pipeline_outputs = {
    "dino_boxes_pixels": boxes_pixels,
    "dino_phrases": phrases,
    "selected_indices": selected_indices,
    "selected_similarities": selected_similarities,
    "merged_box_pixels": merged_box,
    "sam_mask": mask,
    "result_path": result_path,
    "annotated_image_rgb": composited
}

# Optionally display inline in Colab
try:
    from google.colab.patches import cv2_imshow
    print("[INFO] Displaying result inline...")
    cv2_imshow(cv2.cvtColor(composited, cv2.COLOR_RGB2BGR))
except:
    print("[INFO] cv2_imshow not available; saved to", result_path)

# You can now download the result file:
# files.download(result_path)


Upload your image (choose file from local).


Saving Screenshot 2025-08-10 at 11.55.29 PM.png to Screenshot 2025-08-10 at 11.55.29 PM (1).png
[INFO] Using device: cpu
[INFO] Loading GroundingDINO model (this may take a while on CPU)...


final text_encoder_type: bert-base-uncased


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


[INFO] GroundingDINO loaded.
[INFO] Loading CLIP model...
[INFO] CLIP loaded.
[INFO] Building SAM model (vit_h)...
[INFO] SAM loaded.
[INFO] Reading image: Screenshot 2025-08-10 at 11.55.29 PM (1).png
[INFO] Running GroundingDINO inference...


[INFO] GroundingDINO returned 3 boxes.
[INFO] Scoring DINO crops with CLIP against: vendor with vegetables
[INFO] Selected 0 boxes above CLIP threshold 0.25.
[WARN] No boxes passed threshold — selecting top 2 by similarity as fallback.
[INFO] Fallback selected indices: []
[WARN] Still no boxes to merge. Saving original image and exiting.
[DONE] Saved original image at /content/combined_result.jpg


SystemExit: 

In [1]:
# ================================
# GroundingDINO + SAM + CLIP Pipeline
# ================================

import torch
import cv2
import numpy as np
from groundingdino.util.inference import load_model, load_image, predict, annotate
from segment_anything import sam_model_registry, SamPredictor
import clip
from PIL import Image

# ---------------- SETTINGS ----------------
image_path = "/content/my_image.png"  # Change to your image path
prompt_text = "vendor and vegetables" # Change prompt

# Model paths (update if needed)
dino_config = "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"
dino_weights = "/content/GroundingDINO/weights/groundingdino_swint_ogc.pth"
sam_checkpoint = "/content/sam_vit_h_4b8939.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- LOAD MODELS ----------------
# GroundingDINO
dino_model = load_model(dino_config, dino_weights).to(device)

# SAM
sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint).to(device)
sam_predictor = SamPredictor(sam)

# CLIP
clip_model, preprocess = clip.load("ViT-B/32", device=device)

# ---------------- DINO DETECTION ----------------
image_source, image_tensor = load_image(image_path)

boxes, logits, phrases = predict(
    model=dino_model,
    image=image_tensor,
    caption=prompt_text,
    box_threshold=0.35,
    text_threshold=0.25,
    device=device
)

# ---------------- SAM SEGMENTATION ----------------
image_bgr = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
sam_predictor.set_image(image_rgb)

masks = []
for box in boxes:
    box_np = box.cpu().numpy()
    mask, _, _ = sam_predictor.predict(
        point_coords=None,
        point_labels=None,
        box=box_np,
        multimask_output=False
    )
    masks.append(mask[0])

# ---------------- CLIP MERGING ----------------
# Encode detected regions
image_crops = []
for mask in masks:
    # Get bounding box from mask
    y_indices, x_indices = np.where(mask)
    x_min, x_max = x_indices.min(), x_indices.max()
    y_min, y_max = y_indices.min(), y_indices.max()
    crop = image_rgb[y_min:y_max, x_min:x_max]
    image_crops.append(Image.fromarray(crop))

image_features = []
with torch.no_grad():
    for crop in image_crops:
        crop_tensor = preprocess(crop).unsqueeze(0).to(device)
        features = clip_model.encode_image(crop_tensor)
        image_features.append(features / features.norm(dim=-1, keepdim=True))

# Encode the "combined" prompt for grouping
text_features = clip_model.encode_text(
    clip.tokenize(prompt_text).to(device)
)
text_features /= text_features.norm(dim=-1, keepdim=True)

# Find which regions are related
related_boxes = []
for i, feat in enumerate(image_features):
    similarity = (feat @ text_features.T).item()
    if similarity > 0.25:  # Threshold for grouping
        related_boxes.append(boxes[i])

# Merge related boxes into one
if related_boxes:
    x_min = min(b[0] for b in related_boxes)
    y_min = min(b[1] for b in related_boxes)
    x_max = max(b[2] for b in related_boxes)
    y_max = max(b[3] for b in related_boxes)
    merged_box = torch.tensor([x_min, y_min, x_max, y_max]).unsqueeze(0)
else:
    merged_box = boxes

# ---------------- ANNOTATE & SAVE ----------------
annotated_image = annotate(image_source, merged_box, torch.tensor([1.0]), [prompt_text])
output_path = "/content/final_combined.jpg"
cv2.imwrite(output_path, annotated_image)
print(f"Saved result at {output_path}")


final text_encoder_type: bert-base-uncased


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


KeyboardInterrupt: 